# Large Language Models Overview

## Learning Objectives
- Understand what large language models (LLMs) are.
- Learn about their architecture, training, and capabilities.

## Introduction
LLMs are deep learning models trained on massive text corpora to generate or understand human language, driving advances in NLP.

### The Transformer
The transformer is a network architecture solely based on attention mechanisms
without any convolutional or recurrent layer. It is a type of encoder-decoder system.

Transformers have been widely used in language, vision, speech, and reinforcement learning

### Prompt Engineering
In generative AI, the input information is optimized by using prompt engineering which is the structuring of inputs in a way that will be the most beneficial for the output. The model’s responses can be made more useful by modifying the context and choice of words used to make the prompt.

## Core Concepts

- Transformer Architecture: This is the neural network design that underpins nearly all state-of-the-art LLMs. Its core component, the self-attention mechanism, allows it to efficiently process long sequences of text and capture complex relationships between words, making it highly scalable and powerful.

- Pretraining: This is the initial, computationally massive training phase. The model is trained on a vast corpus of text from the internet and books in a self-supervised manner. It learns by performing a simple task that doesn't require manual labels, most commonly "next-word prediction." By doing this at an immense scale, the model is forced to learn grammar, facts, reasoning abilities, and a general "world model."

- Fine-tuning: After pretraining, the model is a generalist. Fine-tuning is the process of adapting this general model to a specific task. This is done by continuing the training on a much smaller, curated dataset. For example, a model can be fine-tuned on a dataset of conversations to become a chatbot, or on a dataset of code to become a programming assistant.

- Scaling Laws: These are empirical findings that have been a major driving force in the field. They show that a model's performance on various tasks improves predictably as you increase the model size (number of parameters), the dataset size, and the compute budget for training. This discovery provided a clear justification for building ever-larger models, leading to the powerful LLMs we have today.

## Example
Summary of GPT and BERT model differences.

In [5]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import os

# --- 1. Download a small text dataset ---
path_to_file = keras.utils.get_file('shakespeare.txt', 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt')

# Read, then decode for py2 compat.
text = open(path_to_file, 'rb').read().decode(encoding='utf-8')
print(f'Length of text: {len(text)} characters')

# The unique characters in the file
vocab = sorted(set(text))
print(f'{len(vocab)} unique characters')

# --- 2. Process the text ---
# Create a mapping from unique characters to indices
char2idx = {u:i for i, u in enumerate(vocab)}
idx2char = np.array(vocab)

text_as_int = np.array([char2idx[c] for c in text])

# The maximum length sentence we want for a single input in characters
seq_length = 100
examples_per_epoch = len(text)//(seq_length+1)

# Create training examples / targets
char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
sequences = char_dataset.batch(seq_length+1, drop_remainder=True)

def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text

dataset = sequences.map(split_input_target)

# --- 3. Create the Training Batches ---
BATCH_SIZE = 64
BUFFER_SIZE = 10000

dataset = (
    dataset
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE, drop_remainder=True))

print("Dataset prepared. Example batch:")
for input_example, target_example in dataset.take(1):
    print("Input shape:", input_example.shape)
    print("Target shape:", target_example.shape)


# --- 4. Build The Model ---
vocab_size = len(vocab)
embedding_dim = 256
rnn_units = 1024

def build_model(vocab_size, embedding_dim, rnn_units, batch_size):
    inputs = tf.keras.Input(batch_shape=[batch_size, None])
    x = tf.keras.layers.Embedding(vocab_size, embedding_dim)(inputs)
    x = tf.keras.layers.GRU(rnn_units,
                            return_sequences=True,
                            stateful=True,
                            recurrent_initializer='glorot_uniform')(x)
    outputs = tf.keras.layers.Dense(vocab_size)(x)
    
    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    return model

model = build_model(
    vocab_size=len(vocab),
    embedding_dim=embedding_dim,
    rnn_units=rnn_units,
    batch_size=BATCH_SIZE)

model.summary()

# --- 5. Train the Model ---
def loss(labels, logits):
    return tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)

model.compile(optimizer='adam', loss=loss)

# Directory where the checkpoints will be saved
checkpoint_dir = './training_checkpoints'
filepath = os.path.join(checkpoint_dir, "ckpt_{epoch}.weights.h5")

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=filepath,
    save_weights_only=True)

print("\n--- Starting Training ---")
EPOCHS = 10
history = model.fit(dataset, epochs=EPOCHS, callbacks=[checkpoint_callback])
print("--- Training Finished ---\n")


# --- 6. Generate Text ---
print("Finding latest checkpoint...")

# Build the model for generation (batch_size=1)
model = build_model(vocab_size, embedding_dim, rnn_units, batch_size=1)

# Check if checkpoint directory exists and has checkpoint files
if os.path.exists(checkpoint_dir) and any(f.endswith('.weights.h5') for f in os.listdir(checkpoint_dir)):
    checkpoint_files = [f for f in os.listdir(checkpoint_dir) if f.startswith('ckpt_') and f.endswith('.weights.h5')]
    latest_epoch = max([int(f.split('_')[1].split('.')[0]) for f in checkpoint_files])
    latest_checkpoint_file = os.path.join(checkpoint_dir, f"ckpt_{latest_epoch}.weights.h5")
    
    print(f"Loading weights from: {latest_checkpoint_file}")
    model.load_weights(latest_checkpoint_file)
else:
    print("WARNING: No checkpoints found. Text will be generated from an untrained model.")


def generate_text(model, start_string, num_generate=1000):
    input_eval = [char2idx[s] for s in start_string]
    input_eval = tf.expand_dims(input_eval, 0)
    text_generated = []
    temperature = 1.0

    # CORRECTED: The model's state is already reset because we built a new model instance.
    # model.reset_states() # This line is removed.

    for i in range(num_generate):
        predictions = model(input_eval)
        predictions = tf.squeeze(predictions, 0)
        predictions = predictions / temperature
        predicted_id = tf.random.categorical(predictions, num_samples=1)[-1,0].numpy()
        input_eval = tf.expand_dims([predicted_id], 0)
        text_generated.append(idx2char[predicted_id])

    return (start_string + ''.join(text_generated))

# --- Generate text and print it out ---
print("\n--- Generating Text ---")
print(generate_text(model, start_string=u"ROMEO: "))

Length of text: 1115394 characters
65 unique characters
Dataset prepared. Example batch:
Input shape: (64, 100)
Target shape: (64, 100)


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (64, None)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_5 (Embedding)         │ (64, None, 256)        │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_4 (GRU)                     │ (64, None, 1024)       │     3,938,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (64, None, 65)         │        66,625 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,021,569 (15.34 MB)

 Trainable params: 4,021,569 (15.34 MB)

 Non-trainable params: 0 (0.00 B)


--- Starting Training ---
Epoch 1/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 15s 78ms/step - loss: 2.5229
Epoch 2/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 14s 77ms/step - loss: 1.8459
Epoch 3/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 14s 77ms/step - loss: 1.6122
Epoch 4/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 14s 77ms/step - loss: 1.4924
Epoch 5/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 14s 77ms/step - loss: 1.4196
Epoch 6/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 14s 77ms/step - loss: 1.3686
Epoch 7/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 14s 77ms/step - loss: 1.3281
Epoch 8/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 14s 79ms/step - loss: 1.2931
Epoch 9/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 14s 78ms/step - loss: 1.2608
Epoch 10/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 14s 78ms/step - loss: 1.2290
--- Training Finished ---

Finding latest checkpoint...
Loading weights from: ./training_checkpoints/ckpt_10.weights.h5

--- Generating Text ---
ROMEO: you shall be! Henry all Romeo here, may shunly she sac?
Come, my good envoinius, is your hind from Russion;
To be content, such as

## Exercise
Research a large language model and summarize its features and applications.

## Summary
- LLMs have transformed natural language understanding and generation.
- Their scale and architecture enable powerful downstream applications.


## Further Reading
- [GPT Papers](https://openai.com/research)
- [BERT Paper](https://arxiv.org/abs/1810.04805)
- [Transformer Architecture](https://arxiv.org/abs/1706.03762)
